# Sentiment Labeling Pipeline

**Steps:**

1. Load `datasets/final/mda_shared_preprocessed.parquet`, explode `sentences`
2. Drop duplicate sentences, then create a randomized **400-sentence** manual sample with positive/negative/neutral coverage
3. Sample **50k sentences** → LLM few-shot labels (inter-rater check)
4. Merge with your manual labels → Cohen's Kappa
   - κ > 0.7 → proceed; else revise prompt in Section 3
5. Label **entire dataset** with LLM
6. Check class proportions (>80% dominant → prompt-refinement placeholder)
7. Spot-check 100 random sentences → Excel
8. Save final labeled dataset

**Labeling guidelines:**

- Clear growth signal / positive factual outcome → **positive**
- Negative factual outcome → **negative**
- Forward-looking statements → **neutral**
- Backward-looking negative statements → **negative**
- Otherwise → **neutral**


## 0. Imports & Config


In [1]:
%pip install -U "typing_extensions>=4.12.2" "pydantic>=2.7" "pydantic-core>=2.18" "tqdm" "vllm"
print("If this cell upgraded packages, restart the kernel before running imports.")


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
If this cell upgraded packages, restart the kernel before running imports.


In [2]:
import os
import random
import re

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from tqdm.auto import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# NOTE: vLLM requires a CUDA GPU. Install with: pip install vllm

# -- Paths --
DATA_PATH = "datasets/final/mda_shared_preprocessed.parquet"
MANUAL_OUT = "sentiment_analysis/manual_labeling_sample.xlsx"
MANUAL_LABELED_PATH = "sentiment_analysis/manual_labeling_sample_LABELED.xlsx"
FINAL_OUT = "sentiment_analysis/labeled_dataset.csv"
SPOT_CHECK_OUT = "sentiment_analysis/spot_check_100.xlsx"

# -- GPU config --
# tensor_parallel_size: number of GPUs to shard the model across.
# Reads CUDA_VISIBLE_DEVICES if set by the cluster scheduler, otherwise uses all visible GPUs.
_cuda_devs = os.environ.get("CUDA_VISIBLE_DEVICES", "")
NUM_GPUS = len(_cuda_devs.split(",")) if _cuda_devs else 1

# Sentences per chunk when calling llm.generate(). vLLM batches internally,
# but chunking lets tqdm show progress for very large labeling runs.
CHUNK_SIZE = 8_192

VALID_LABELS = {"positive", "negative", "neutral"}
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


ModuleNotFoundError: No module named 'vllm'

## 1. Load Parquet & Explode Sentences


In [ ]:
df_raw = pd.read_parquet(DATA_PATH)
print(f"✅ Raw filings loaded: {len(df_raw):,}")

required_cols = [
    "doc_id",
    "company_name",
    "filing_type",
    "filing_date",
    "year",
    "quarter",
    "sentences",
]
missing = [c for c in required_cols if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = (
    df_raw[required_cols]
    .explode("sentences")
    .rename(columns={"sentences": "sentence"})
    .dropna(subset=["sentence"])
    .reset_index(drop=True)
)
df["sentence"] = df["sentence"].astype(str).str.strip()
df = df[df["sentence"].str.len() > 20].reset_index(drop=True)

print(f"✅ Total sentences (before dedup): {len(df):,}")
df.head(3)


## 2. Deduplicate + Randomized 400-Sentence Sample (with Class Coverage)


In [ ]:
# Show which sentences appear more than once (before deduplication)
dup_mask = df.duplicated(subset=["sentence"], keep=False)
dups = df[dup_mask]

print(f"✅ Total rows before dedup: {len(df):,}")
print(
    f"✅ Rows in duplicate groups: {len(dups):,} ({dups['sentence'].nunique():,} unique texts)"
)

if dups.empty:
    print("No duplicate sentences found.")
else:
    counts = dups.groupby("sentence").size().sort_values(ascending=False)
    print("\nTop 5 most-repeated sentences:")
    for i, (sent, cnt) in enumerate(counts.head(5).items(), start=1):
        print(f"  {i}. {cnt}x | {sent[:120]}{'...' if len(sent) > 120 else ''}")

    preview = (
        dups[dups["sentence"].isin(counts.head(5).index)][
            ["sentence", "doc_id", "company_name", "filing_date", "filing_type"]
        ]
        .sort_values(["sentence", "company_name", "filing_date"])
        .head(20)
    )
    print("\nExample duplicate rows:")
    display(preview)


In [ ]:
MANUAL_N = 400

# Deduplicate
before_dedup = len(df)
df = df.drop_duplicates(subset=["sentence"]).reset_index(drop=True)
after_dedup = len(df)

# Heuristic buckets to ensure all 3 classes appear in the manual sample
POS_HINTS = (
    "increased",
    "increase",
    "improved",
    "improvement",
    "growth",
    "grew",
    "strong",
    "record",
    "higher",
    "gain",
    "expanded",
    "expansion",
    "profit",
    "profitable",
    "beat",
    "outperformed",
)
NEG_HINTS = (
    "decreased",
    "decrease",
    "declined",
    "decline",
    "lower",
    "loss",
    "impairment",
    "charge",
    "charges",
    "restructuring",
    "adverse",
    "weak",
    "headwind",
    "downturn",
    "reduced",
    "fell",
    "drop",
)
NEUTRAL_HINTS = (
    "expect",
    "expects",
    "expected",
    "anticipate",
    "may",
    "might",
    "could",
    "will",
    "plan",
    "plans",
    "guidance",
    "outlook",
    "forecast",
)


def heuristic_bucket(sentence: str) -> str:
    s = sentence.lower()
    if any(w in s for w in NEUTRAL_HINTS):
        return "neutral"
    pos = any(w in s for w in POS_HINTS)
    neg = any(w in s for w in NEG_HINTS)
    if pos and not neg:
        return "positive"
    if neg and not pos:
        return "negative"
    return "neutral"


df_pool = df.copy()
df_pool["sample_bucket"] = df_pool["sentence"].apply(heuristic_bucket)

# Sample ~equal numbers from each bucket, then shuffle
target_each = MANUAL_N // 3
parts = []
for label in ["positive", "negative", "neutral"]:
    group = df_pool[df_pool["sample_bucket"] == label]
    take_n = min(target_each, len(group))
    if take_n > 0:
        parts.append(group.sample(n=take_n, random_state=SEED))

manual_sample = (
    pd.concat(parts, ignore_index=True) if parts else df_pool.iloc[0:0].copy()
)

# Fill any shortfall with random remaining rows
shortfall = MANUAL_N - len(manual_sample)
if shortfall > 0:
    remaining = df_pool[~df_pool.index.isin(manual_sample.index)]
    extra = remaining.sample(n=min(shortfall, len(remaining)), random_state=SEED)
    manual_sample = pd.concat([manual_sample, extra], ignore_index=True)

manual_sample = manual_sample.sample(frac=1, random_state=SEED).reset_index(drop=True)
manual_sample["manual_label"] = ""

print(f"✅ Duplicates removed: {before_dedup - after_dedup:,}")
print("✅ Sample bucket counts:")
print(manual_sample["sample_bucket"].value_counts().to_string())

if os.path.exists(MANUAL_OUT):
    print(
        f"\n⚠️  '{MANUAL_OUT}' already exists — skipping write to preserve your labels."
    )
else:
    manual_sample.drop(columns=["sample_bucket"]).to_excel(MANUAL_OUT, index=False)
    print(f"\n✅ Manual sample written to '{MANUAL_OUT}'.")

manual_sample[
    ["sentence", "sample_bucket", "company_name", "year", "manual_label"]
].head(8)


## 3. LLM Few-Shot Labeling — 50k Sentences (Inter-rater Check)


In [ ]:
# Qwen2.5-1.5B-Instruct-AWQ: open-weight, INT4-quantized (~1 GB per GPU)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"

llm = LLM(
    model=MODEL_ID,
    quantization="awq",
    max_model_len=1024,  # system + few-shot + sentence ≈ 500 tokens; 512 was too tight
    tensor_parallel_size=NUM_GPUS,  # shard across all available GPUs
    gpu_memory_utilization=0.95,  # use 95% of VRAM (default: 0.90)
    enable_prefix_caching=True,  # cache KV for the shared system+few-shot prefix — biggest speedup
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
sampling_params = SamplingParams(temperature=0, max_tokens=5)

print(f"✅ LLM loaded on {NUM_GPUS} GPU(s) with prefix caching enabled")

# Edit these examples or rules if Cohen's kappa < 0.7 in Section 4
SYSTEM_PROMPT = (
    "You are a financial sentiment classifier for SEC 10-K/10-Q filings. "
    "Classify the sentence as exactly one of: positive, negative, neutral. "
    "positive = clear factual growth (revenue up, profit grew, margin expanded). "
    "negative = clear factual decline (revenue fell, loss, impairment, charges). "
    "negative = growth below expectations (lower than expected, missed targets). "
    "neutral  = forward-looking/hedging (expect, may, could, will, plan). "
    "neutral  = mixed signals, boilerplate, or ambiguous sentences. "
    "Respond with ONLY the label word."
)

FEW_SHOT = (
    'Sentence: "Revenue increased 18% driven by strong demand." Label: positive\n'
    'Sentence: "Net income declined 32% due to higher costs." Label: negative\n'
    'Sentence: "We expect continued growth next fiscal year." Label: neutral\n'
    'Sentence: "Gross margin decreased significantly in fiscal 2022." Label: negative\n'
    'Sentence: "Growth was lower than expected in Q3." Label: negative\n'
    'Sentence: "Operating cash flow improved by $450 million." Label: positive\n'
    'Sentence: "We may face increased competition." Label: neutral\n'
    'Sentence: "The following discusses our results of operations." Label: neutral\n'
)


def _build_prompt(sentence: str) -> str:
    user_msg = FEW_SHOT + f'Sentence: "{sentence}" Label:'
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )


# Sentences containing these words are almost always neutral — skip the LLM for speed
_HEDGE_TOKENS = frozenset(
    [
        "expect",
        "expects",
        "expected",
        "anticipate",
        "anticipates",
        "may",
        "might",
        "could",
        "would",
        "will",
        "plan",
        "plans",
        "guidance",
        "outlook",
        "forecast",
        "intend",
        "intends",
        "believe",
        "believes",
        "project",
        "projects",
        "should",
    ]
)


def _is_obvious_neutral(sentence: str) -> bool:
    tokens = set(re.sub(r"[^a-z ]", " ", sentence.lower()).split())
    return bool(tokens & _HEDGE_TOKENS)


def llm_label_batch(sentences: list[str]) -> list[str]:
    """Label a list of sentences. Returns one label per sentence."""
    n = len(sentences)
    output = ["neutral"] * n  # default to neutral

    # Pre-filter obvious neutrals — no GPU needed for these
    llm_idx, llm_sents = [], []
    for i, s in enumerate(sentences):
        if not _is_obvious_neutral(s):
            llm_idx.append(i)
            llm_sents.append(s)
    print(
        f"  ✅ Pre-filtered {n - len(llm_sents):,} neutrals; sending {len(llm_sents):,} to LLM"
    )

    if not llm_sents:
        return output

    # Build prompts (shared prefix means prefix caching kicks in for all of them)
    prompts = [_build_prompt(s) for s in llm_sents]

    # Process in chunks so tqdm can show progress for large runs
    raw_results = []
    for start in tqdm(
        range(0, len(prompts), CHUNK_SIZE), desc="LLM labeling", unit="chunk"
    ):
        raw_results.extend(
            llm.generate(prompts[start : start + CHUNK_SIZE], sampling_params)
        )

    for orig_i, out in zip(llm_idx, raw_results):
        text = out.outputs[0].text.strip().lower()
        output[orig_i] = text if text in VALID_LABELS else "neutral"

    print(f"  ✅ Done. Label counts: {pd.Series(output).value_counts().to_dict()}")
    return output


In [ ]:
LLMCHECK_N = 50_000

# Include all manual sentences so we can compute kappa later
manual_idx = set(manual_sample.index)
rest = df.loc[~df.index.isin(manual_idx)]
extra_n = min(LLMCHECK_N - len(manual_sample), len(rest))

llm_check = pd.concat(
    [
        manual_sample.drop(columns=["manual_label", "sample_bucket"]),
        rest.sample(n=extra_n, random_state=SEED),
    ],
    ignore_index=True,
)

print(f"✅ LLM inter-rater sample prepared: {len(llm_check):,} sentences")
print("✅ Labeling with LLM...")
llm_check["llm_label"] = llm_label_batch(llm_check["sentence"].tolist())
print("✅ Done.\n", llm_check["llm_label"].value_counts().to_string())


## 4. Merge Manual Labels & Compute Cohen's Kappa

> **Action required before running this cell:**  
> Open `sentiment_analysis/manual_labeling_sample.xlsx`, fill the `manual_label` column  
> (values: `positive` / `negative` / `neutral`), then save as:  
> `sentiment_analysis/manual_labeling_sample_LABELED.xlsx`


In [ ]:
if not os.path.exists(MANUAL_LABELED_PATH):
    raise FileNotFoundError(
        f"Not found: {MANUAL_LABELED_PATH}\n"
        "Complete manual labeling first (see instructions above)."
    )

manual_labeled = pd.read_excel(MANUAL_LABELED_PATH)
manual_labeled["manual_label"] = manual_labeled["manual_label"].str.strip().str.lower()
manual_labeled = manual_labeled[
    manual_labeled["manual_label"].isin(VALID_LABELS)
].copy()
print(f"✅ Valid manually labeled rows: {len(manual_labeled):,}")

merged = manual_labeled.merge(
    llm_check[["sentence", "llm_label"]], on="sentence", how="inner"
)
print(f"✅ Matched for kappa: {len(merged):,} rows")

kappa = cohen_kappa_score(merged["manual_label"], merged["llm_label"])
print(f"\n✅ Cohen's Kappa = {kappa:.4f}")

if kappa > 0.7:
    print("✅ Good agreement (κ > 0.7) — proceed to full labeling.")
else:
    print(
        "⚠️  Poor agreement (κ ≤ 0.7) — revise the few-shot prompt in Section 3 and re-run."
    )

pd.crosstab(
    merged["manual_label"],
    merged["llm_label"],
    rownames=["manual"],
    colnames=["llm"],
    margins=True,
)


## 5. Label Entire Dataset

Runs only if kappa > 0.7.


In [ ]:
assert kappa > 0.7, f"Kappa = {kappa:.4f} ≤ 0.7. Revise prompt before proceeding."

# Re-use already-labeled sentences to avoid re-running the LLM on them
cached = llm_check.set_index("sentence")["llm_label"].to_dict()
df_full = df.copy()
df_full["llm_label"] = df_full["sentence"].map(cached)

unlabeled = df_full["llm_label"].isna()
to_label = df_full.loc[unlabeled, "sentence"].tolist()
print(
    f"✅ Already labeled (cached): {(~unlabeled).sum():,} | Remaining: {len(to_label):,}"
)

if to_label:
    print("✅ Labeling remaining sentences...")
    df_full.loc[unlabeled, "llm_label"] = llm_label_batch(to_label)

print(f"\n✅ Full dataset labeled: {len(df_full):,} sentences")
df_full["llm_label"].value_counts()


## 6. Check Class Proportions


In [ ]:
props = df_full["llm_label"].value_counts(normalize=True)
print("✅ Class proportions:")
print(props.map("{:.2%}".format).to_string())

dom_cls = props.idxmax()
dom_pct = props.max()

if dom_pct > 0.80:
    print(f"\n⚠️  '{dom_cls}' dominates at {dom_pct:.1%}. Consider:")
    print("  1. Add more contrastive few-shot examples for minority classes")
    print("  2. Tighten/loosen the dominant-class definition in the prompt")
    print("  3. Re-run Section 3 with the revised prompt and re-check kappa")
else:
    print(f"\n✅ Class distribution looks healthy (no class > 80%).")


## 7. Spot-Check 100 Random Sentences


In [ ]:
spot = df_full.sample(n=100, random_state=SEED)[
    ["sentence", "company_name", "year", "filing_type", "llm_label"]
].reset_index(drop=True)
spot["reviewer_label"] = ""  # optional: fill in to verify quality

spot.to_excel(SPOT_CHECK_OUT, index=False)
print(f"✅ Spot-check sample saved to '{SPOT_CHECK_OUT}'")
spot.head(10)


## 8. Save Final Labeled Dataset


In [ ]:
df_final = df_full[
    [
        "doc_id",
        "company_name",
        "filing_type",
        "filing_date",
        "year",
        "quarter",
        "sentence",
        "llm_label",
    ]
].rename(columns={"llm_label": "sentiment"})
df_final.to_csv(FINAL_OUT, index=False)

print(f"✅ Final dataset saved -> {FINAL_OUT}")
print(f"✅ Shape: {df_final.shape}")
print("\n✅ Label distribution:")
print(df_final["sentiment"].value_counts().to_string())
df_final.head(5)